# 📗 Bài 3 — Attention

**Tuần 14** · ⭐⭐ **Notebook quan trọng nhất Phase 06**

Sau notebook này, kiến trúc Transformer không còn là hộp đen.

In [ ]:
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]
torch.manual_seed(42)
print("PyTorch", torch.__version__)

## 1. Vấn đề cần giải

```
"Con mèo đuổi con chuột vì NÓ đói."
```

`NÓ` là mèo hay chuột? Muốn biết, khi xử lý từ `NÓ` ta phải **nhìn lại** các từ trước và quyết định **chú ý vào từ nào**.

Đó chính xác là điều attention làm.

Ở bài 2, mean pooling cho `"mèo đuổi chuột"` và `"chuột đuổi mèo"` ra **cùng một vector**. Attention sửa điều đó.

## 2. Ẩn dụ: tra cứu trong thư viện

Mỗi token tạo ra **ba** vector từ chính nó:

| Vector | Vai trò | Ẩn dụ |
|---|---|---|
| **Query (Q)** | *"Tôi đang tìm gì?"* | Câu hỏi tra cứu |
| **Key (K)** | *"Tôi chứa thông tin gì?"* | Nhãn trên gáy sách |
| **Value (V)** | *"Nội dung thật của tôi"* | Nội dung trong sách |

```
① Query của token hiện tại so với Key của MỌI token  →  điểm giống nhau
② Softmax các điểm đó                                 →  trọng số (tổng = 1)
③ Trung bình CÓ TRỌNG SỐ các Value                    →  đầu ra
```

## 3. ⭐ Attention bằng code — bốn dòng

In [ ]:
def attention(Q, K, V, mask=None):
    d = Q.shape[-1]

    diem = Q @ K.transpose(-2, -1) / math.sqrt(d)       # ① so Q với mọi K
    if mask is not None:
        diem = diem.masked_fill(mask == 0, float("-inf"))
    trong_so = torch.softmax(diem, dim=-1)              # ② softmax
    return trong_so @ V, trong_so                       # ③ trung bình có trọng số


torch.manual_seed(0)
Q = K = V = torch.randn(1, 4, 8)
out, w = attention(Q, K, V)

print(f"  Q, K, V shape : {tuple(Q.shape)}")
print(f"  đầu ra        : {tuple(out.shape)}")
print(f"  trọng số      : {tuple(w.shape)}")
print(f"  mỗi hàng tổng = 1? {torch.allclose(w.sum(-1), torch.ones(1, 4))}")

$$\text{Attention} = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d}}\right)V$$

Chỉ có vậy. Bốn phép: nhân ma trận, chia, softmax, nhân ma trận.

**Đây là trái tim của mọi LLM.**

## 4. ⚠️ Vì sao phải chia cho √d?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

for ax, d in zip(axes, [8, 64, 512]):
    torch.manual_seed(0)
    q, k = torch.randn(1, 6, d), torch.randn(1, 6, d)
    diem = q @ k.transpose(-2, -1)

    w_khong_chia = torch.softmax(diem, dim=-1)[0]
    w_co_chia = torch.softmax(diem / math.sqrt(d), dim=-1)[0]

    ax.plot(w_khong_chia[0], "o-", color=MAU_NHAN, label="KHÔNG chia √d")
    ax.plot(w_co_chia[0], "o-", color=MAU_CHINH, label="CÓ chia √d")
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"d = {d}", fontsize=10)
    ax.set_xlabel("token")
    if d == 8:
        ax.set_ylabel("trọng số attention")
    ax.legend(frameon=False, fontsize=8)

plt.suptitle("Trọng số attention của token đầu tiên", y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
print(f"  {'d':>5} {'max không chia':>16} {'max có chia':>14}")
print("  " + "-" * 38)
for d in [8, 64, 256, 512]:
    torch.manual_seed(0)
    q, k = torch.randn(1, 6, d), torch.randn(1, 6, d)
    diem = q @ k.transpose(-2, -1)
    print(f"  {d:>5} {torch.softmax(diem, -1).max():>16.4f} "
          f"{torch.softmax(diem/math.sqrt(d), -1).max():>14.4f}")

**Không chia √d, softmax trở nên gần như one-hot khi `d` lớn.**

**Vì sao điều đó tệ?** Đạo hàm của softmax ở vùng bão hoà gần bằng 0 → **gradient biến mất** → model không học được gì.

**Giải thích toán học:** nếu các phần tử của Q và K độc lập, kỳ vọng 0, phương sai 1, thì tích vô hướng `Q·K` có phương sai bằng `d`. Với `d = 512`, độ lệch chuẩn là ~22.6 — các điểm nằm rải từ −70 đến +70. Chia cho `√d` đưa phương sai về 1.

Đó là chữ **"scaled"** trong *scaled dot-product attention*.

## 5. ⭐ Nhìn thấy attention — ví dụ có ý nghĩa

In [ ]:
cau = ["Con", "mèo", "đuổi", "con", "chuột", "vì", "nó", "đói"]
T = len(cau)

# Đặt TAY một ma trận attention "lý tưởng" để thấy ý nghĩa
w_thu_cong = torch.eye(T) * 0.5
w_thu_cong[6, 1] = 0.9        # "nó"   -> "mèo"
w_thu_cong[2, 1] = 0.6        # "đuổi" -> "mèo" (chủ ngữ)
w_thu_cong[2, 4] = 0.5        # "đuổi" -> "chuột" (tân ngữ)
w_thu_cong[7, 6] = 0.7        # "đói"  -> "nó"
w_thu_cong = w_thu_cong / w_thu_cong.sum(dim=1, keepdim=True)


def ve_attention(w, nhan, tieu_de):
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=w.max())
    ax.set_xticks(range(len(nhan)), nhan, rotation=45, ha="right")
    ax.set_yticks(range(len(nhan)), nhan)
    ax.set_xlabel("chú ý VÀO token nào"); ax.set_ylabel("token đang HỎI")
    ax.grid(visible=False)
    for i in range(len(nhan)):
        for j in range(len(nhan)):
            if w[i, j] > 0.05:
                ax.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if w[i, j] > w.max() * .6 else "#222")
    plt.colorbar(im, ax=ax, shrink=.8)
    plt.title(tieu_de); plt.tight_layout(); plt.show()


ve_attention(w_thu_cong, cau, 'Ma trận attention "lý tưởng"')
print('  Hàng "nó" chú ý mạnh vào "mèo" -> model đã giải được đại từ.')

> 🔑 Mỗi **hàng** là một token đang hỏi *"tôi nên chú ý vào ai?"*. Tổng mỗi hàng bằng 1 (do softmax).
>
> Ma trận này là **công cụ chẩn đoán mạnh nhất** khi debug Transformer. Ở bài 4 bạn sẽ vẽ nó từ mini-GPT thật của mình.

## 6. ⭐⭐ Causal mask — thứ làm nên GPT

In [ ]:
def tao_causal_mask(T):
    return torch.tril(torch.ones(T, T))


m = tao_causal_mask(6)
print("Causal mask (1 = được nhìn, 0 = bị che):")
print(m.int().numpy())

In [ ]:
torch.manual_seed(0)
Q = K = V = torch.randn(1, 6, 16)

_, w_tu_do = attention(Q, K, V)
_, w_nhan_qua = attention(Q, K, V, mask=tao_causal_mask(6))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, w, ten in [(a1, w_tu_do[0], "KHÔNG mask (BERT)"),
                   (a2, w_nhan_qua[0], "CÓ causal mask (GPT)")]:
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xlabel("chú ý vào"); ax.set_ylabel("token đang hỏi")
    ax.set_title(ten); ax.grid(visible=False)
    for i in range(6):
        for j in range(6):
            ax.text(j, i, f"{w[i,j]:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if w[i, j] > .5 else "#222")
plt.tight_layout(); plt.show()

print(f"  Hàng đầu (có mask): {w_nhan_qua[0, 0].numpy().round(2)}")
print("  -> Token đầu tiên CHỈ nhìn được chính nó.")

**Một dòng `masked_fill` tạo ra khác biệt giữa hai họ model thống trị NLP:**

| Kiến trúc | Mask | Nhìn được | Ví dụ | Dùng để |
|---|---|---|---|---|
| **Decoder-only** | Có | Chỉ quá khứ | GPT, Claude, Llama | **Sinh** văn bản |
| **Encoder-only** | Không | Cả hai phía | BERT, PhoBERT | **Hiểu** văn bản |

**Vì sao không để model nhìn tương lai khi train?**

Vì lúc **sinh** thật, tương lai chưa tồn tại. Nếu train có nhìn tương lai, model sẽ học cách "gian lận" — dự đoán token thứ `i` bằng cách nhìn chính token thứ `i`. Loss rất thấp khi train nhưng model **hoàn toàn vô dụng** khi sinh.

> 🔗 Đây là một dạng **data leakage** (Phase 04) ở cấp độ kiến trúc.

**Lợi ích phụ rất lớn:** nhờ mask, **một** lần forward trên chuỗi dài `T` cho ta `T` bài toán dự đoán cùng lúc (dự đoán token 2 từ token 1, token 3 từ 1–2, …). Đó là lý do train GPT hiệu quả đến vậy — **mọi token trong kho ngữ liệu đều là một ví dụ huấn luyện**.

## 7. Multi-head attention

In [ ]:
def chia_dau(x, so_dau):
    B, T, d = x.shape
    return x.reshape(B, T, so_dau, d // so_dau).transpose(1, 2)


def gop_dau(x):
    B, H, T, dk = x.shape
    return x.transpose(1, 2).contiguous().reshape(B, T, H * dk)


x = torch.randn(2, 6, 12)
print(f"  x            {tuple(x.shape)}")
print(f"  chia 4 đầu   {tuple(chia_dau(x, 4).shape)}   (B, số_đầu, T, d_k)")
print(f"  gộp lại      {tuple(gop_dau(chia_dau(x, 4)).shape)}")
print(f"  gộp(chia(x)) == x ? {torch.allclose(gop_dau(chia_dau(x, 4)), x)}")

In [ ]:
def multi_head(x, Wq, Wk, Wv, Wo, so_dau, nhan_qua=True):
    Q, K, V = (chia_dau(x @ W, so_dau) for W in (Wq, Wk, Wv))
    mask = tao_causal_mask(x.shape[1]) if nhan_qua else None
    ra, w = attention(Q, K, V, mask=mask)
    return gop_dau(ra) @ Wo, w


torch.manual_seed(0)
d, H, T = 16, 4, 8
x = torch.randn(1, T, d)
W = [torch.randn(d, d) * 0.3 for _ in range(4)]

ra, w = multi_head(x, *W, so_dau=H)
print(f"  đầu ra {tuple(ra.shape)}   trọng số {tuple(w.shape)}   ({H} đầu riêng biệt)")

fig, axes = plt.subplots(1, H, figsize=(14, 3.2))
for i, ax in enumerate(axes):
    ax.imshow(w[0, i], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"đầu {i+1}", fontsize=10)
    ax.grid(visible=False); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Mỗi đầu học một kiểu chú ý khác nhau", y=1.05)
plt.tight_layout(); plt.show()

**Vì sao cần nhiều đầu?** Một đầu chỉ học được **một** kiểu quan hệ — vì softmax buộc trọng số phải tổng bằng 1, nên nó phải "chọn" chú ý vào đâu.

Nhiều đầu chạy **song song**, mỗi đầu chuyên môn hoá:
```
đầu 1: chú ý vào chủ ngữ
đầu 2: chú ý vào từ liền trước
đầu 3: chú ý vào động từ chính
đầu 4: chú ý vào dấu câu, ranh giới mệnh đề
```

Các nghiên cứu diễn giải đã tìm thấy những đầu attention chuyên môn thật sự như vậy trong GPT-2 và BERT.

**Vì sao chia `d` thành nhiều phần thay vì mỗi đầu dùng cả `d` chiều?** Để **tổng** chi phí tính toán không đổi. Với `d=768` và 12 đầu, mỗi đầu làm việc với 64 chiều — ta có 12 góc nhìn gần như miễn phí.

**Vì sao cần `Wo` ở cuối?** Sau khi gộp, các đầu chỉ nằm **cạnh nhau** trong vector, chưa "trộn" với nhau. `Wo` cho phép model kết hợp thông tin từ các đầu khác nhau.

## 8. Positional encoding

In [ ]:
def positional_encoding(T, d):
    pe = torch.zeros(T, d)
    pos = torch.arange(T, dtype=torch.float32).unsqueeze(1)
    i = torch.arange(0, d, 2, dtype=torch.float32)
    mau_so = torch.pow(10000.0, i / d)
    pe[:, 0::2] = torch.sin(pos / mau_so)
    pe[:, 1::2] = torch.cos(pos / mau_so)
    return pe


pe = positional_encoding(60, 64)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
im = a1.imshow(pe.T, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
a1.set_xlabel("vị trí trong chuỗi"); a1.set_ylabel("chiều embedding")
a1.set_title("Positional encoding"); a1.grid(visible=False)
plt.colorbar(im, ax=a1, shrink=.8)

for chieu in [0, 4, 16, 40]:
    a2.plot(pe[:, chieu], lw=2, label=f"chiều {chieu}")
a2.set_xlabel("vị trí"); a2.set_ylabel("giá trị")
a2.set_title("Vài chiều — tần số khác nhau"); a2.legend(frameon=False)

plt.tight_layout(); plt.show()

In [ ]:
# Mỗi vị trí có một "vân tay" DUY NHẤT
tuong_dong = (pe @ pe.T) / (pe.norm(dim=1, keepdim=True) @ pe.norm(dim=1, keepdim=True).T)

plt.figure(figsize=(5.5, 4.5))
plt.imshow(tuong_dong, cmap="RdBu_r", vmin=-1, vmax=1)
plt.xlabel("vị trí"); plt.ylabel("vị trí")
plt.title("Cosine giữa các positional encoding")
plt.colorbar(shrink=.8); plt.grid(visible=False)
plt.tight_layout(); plt.show()

print("  Đường chéo sáng = mỗi vị trí giống chính nó nhất.")
print("  Độ sáng giảm dần khi đi xa -> mã hoá được KHOẢNG CÁCH giữa các vị trí.")

**Vì sao Transformer cần positional encoding?**

Attention xử lý mọi token **cùng lúc** và **đối xứng** — nó không có khái niệm thứ tự. Với nó, *"mèo đuổi chuột"* và *"chuột đuổi mèo"* giống hệt nhau.

(Đối lập với RNN: RNN đọc tuần tự nên thứ tự là có sẵn. Đó là thứ Transformer **đánh đổi** để được tính song song — và phải bù lại bằng positional encoding.)

**Vì sao dùng sin/cos mà không đánh số 0, 1, 2, 3…?**
1. Giá trị **bị chặn** trong `[-1, 1]` — vị trí 5000 không áp đảo tín hiệu ngữ nghĩa
2. Mỗi vị trí có một **"vân tay" duy nhất** (tổ hợp nhiều tần số, giống cách số nhị phân dùng các bit)
3. **Quan hệ tương đối** suy ra được: `PE(pos+k)` là một phép quay tuyến tính của `PE(pos)`

| Cách | Ai dùng |
|---|---|
| **Học được** (`nn.Embedding`) | GPT-2, mini-GPT của bạn — đơn giản nhất |
| **Sin/cos** | Transformer gốc (2017) |
| **RoPE** (xoay) | Llama, Claude, hầu hết LLM hiện đại |

> 🔗 Đây là lý do LLM có **context window** giới hạn: model chỉ học cách xử lý vị trí đến một độ dài nhất định. Mọi kỹ thuật mở rộng context (rope scaling…) đều xoay quanh việc sửa phần này.

## 9. Chi phí O(T²)

In [ ]:
do_dai = [128, 512, 2048, 8192, 32768, 128000]
print(f"  {'context':>9} {'ô trong ma trận attention':>28} {'gấp bao nhiêu lần 128':>24}")
print("  " + "-" * 64)
goc = 128 ** 2
for T in do_dai:
    print(f"  {T:>9,} {T*T:>28,} {T*T/goc:>23,.0f}x")

Ma trận attention có kích thước `T × T`. **Chuỗi dài gấp đôi → tính toán gấp bốn**, bộ nhớ cũng gấp bốn.

Đó là lý do:
- Context window dài rất **đắt**
- Có cả một dòng nghiên cứu về *efficient attention*: FlashAttention, sliding window, linear attention
- Số token bạn nhét vào prompt **trực tiếp** quyết định chi phí (Phase 07)

## 10. Tổng kết

```python
def attention(Q, K, V, mask=None):
    d = Q.shape[-1]
    diem = Q @ K.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        diem = diem.masked_fill(mask == 0, float("-inf"))
    w = torch.softmax(diem, dim=-1)
    return w @ V, w
```

| Thành phần | Vai trò |
|---|---|
| **Q, K, V** | Câu hỏi / nhãn / nội dung |
| **√d** | Giữ softmax không bão hoà → gradient chảy |
| **Causal mask** | Cấm nhìn tương lai → sinh được văn bản |
| **Multi-head** | Nhiều kiểu quan hệ song song |
| **Positional encoding** | Bù lại thông tin thứ tự |

**Bài 4 sẽ ghép tất cả lại thành một GPT hoàn chỉnh và train nó.**

## ✍️ Bài tập

```bash
pytest tests/phase06/test_ex03.py -v
```